# 05 · Fine-tuning lightweight SOTA backbones — §5 [20 marks]

Both backbones run at **64×64**, not their native 224 — see `configs/stages/pretrained.yaml`. The resulting handicap is a finding for §6, not a bug.

| | |
|---|---|
| **Owner** | Member 4 |
| **Trains** | `mobilenet_v2`, `squeezenet1_1` — through the same `Trainer` as the custom models |
| **Reads** | the committed split — the same one the custom models use |
| **Writes (official)** | 2 × `history.json` + `test_metrics.json`, curves, confusion matrices |
| **Report needs** | total parameter count and model size (MB), test metrics, fine-tuning strategy, the 64×64 decision, the normalisation choice |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths
from edgecnn.data import build_dataloaders
from edgecnn.evaluation import evaluate_run
from edgecnn.evaluation.curves import plot_loss_curves
from edgecnn.evaluation.reporting import plot_confusion_matrix
from edgecnn.models import build_model_from_config
from edgecnn.models.pretrained import count_trainable_vs_total
from edgecnn.training import Trainer

In [ ]:
# Shared figure style (Member 1): small inline figures, print-quality saved PNGs.
from edgecnn.evaluation.plotting import apply_style

apply_style()

## Config

In [ ]:
BACKBONES = ["mobilenet_v2__adam", "squeezenet1_1__adam"]
configs = {name: load_config(f"configs/experiments/{name}.yaml", mode=MODE) for name in BACKBONES}
for name, c in configs.items():
    print(f"{c.run_id:<30} official={c.is_official}")
configs[BACKBONES[0]].section("pretrained")

## Data — shared by both backbones

Pretrained runs normalise with ImageNet statistics; check `norm_mean` below. State this deviation in §5.

In [ ]:
data = build_dataloaders(configs[BACKBONES[0]])
results, evaluations = {}, {}
print(data.describe(), "| norm_mean", data.norm_mean)

## 1 · MobileNetV2 — `mobilenet_v2__adam`

In [ ]:
cfg = configs["mobilenet_v2__adam"]
model = build_model_from_config(cfg, data.num_classes)
trainable, total = count_trainable_vs_total(model)
print(f"{trainable:,} trainable of {total:,} parameters")

In [ ]:
results["mobilenet_v2"] = Trainer(cfg).fit(model, data, cfg)

In [ ]:
plot_loss_curves(results["mobilenet_v2"], write=cfg.is_official)

In [ ]:
evaluations["mobilenet_v2"] = evaluate_run(cfg, data, results["mobilenet_v2"].checkpoint_path)
print(f"test accuracy {evaluations['mobilenet_v2'].accuracy:.4f}")

In [ ]:
plot_confusion_matrix(evaluations["mobilenet_v2"], write=cfg.is_official)

## 2 · SqueezeNet 1.1 — `squeezenet1_1__adam`

In [ ]:
cfg = configs["squeezenet1_1__adam"]
model = build_model_from_config(cfg, data.num_classes)
trainable, total = count_trainable_vs_total(model)
print(f"{trainable:,} trainable of {total:,} parameters")

In [ ]:
results["squeezenet1_1"] = Trainer(cfg).fit(model, data, cfg)

In [ ]:
plot_loss_curves(results["squeezenet1_1"], write=cfg.is_official)

In [ ]:
evaluations["squeezenet1_1"] = evaluate_run(cfg, data, results["squeezenet1_1"].checkpoint_path)
print(f"test accuracy {evaluations['squeezenet1_1'].accuracy:.4f}")

In [ ]:
plot_confusion_matrix(evaluations["squeezenet1_1"], write=cfg.is_official)

## Notes for §5

- Fine-tuning strategy per backbone, and **both** parameter counts (trainable vs total).
- Why 64×64: cost at the sensor's real resolution. MobileNetV2's 32× stride leaves a 2×2 feature map.
- ImageNet vs EuroSAT normalisation — which was used, and why.
- Different routes to efficiency: inverted residuals (MobileNetV2) vs fire modules (SqueezeNet).

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] both backbones ran on the same machine / runtime
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Guide: COLAB.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [*(paths.metrics_dir(c.run_id) for c in configs.values()),
         *(paths.figures_dir(c.run_id) for c in configs.values())],
        "Add pretrained backbone results (Colab)",
    )